In [5]:
from neo4j import GraphDatabase

# Sesuaikan password Neo4j lokal kamu di sini
URI = "neo4j://localhost:7687"
AUTH = ("neo4j", "utsgrafkoni")

def add_friend(driver, name, friend_name):
    with driver.session() as session:
        session.run(
            "MERGE (a:Person {name: $name}) "
            "MERGE (friend:Person {name: $friend_name}) "
            "MERGE (a)-[:KNOWS]->(friend)",
            name=name, friend_name=friend_name
        )

def print_friends(driver, name):
    with driver.session() as session:
        result = session.run(
            "MATCH (a:Person)-[:KNOWS]->(friend) WHERE a.name = $name "
            "RETURN friend.name ORDER BY friend.name",
            name=name
        )
        for record in result:
            print(record["friend.name"])

with GraphDatabase.driver(URI, auth=AUTH) as driver:
    add_friend(driver, "Arthur", "Guinevere")
    add_friend(driver, "Arthur", "Lancelot")
    add_friend(driver, "Arthur", "Merlin")
    print_friends(driver, "Arthur")

Guinevere
Lancelot
Merlin


Pertanyaan Pendahuluan (Checkpoint)

1. Nama Database yang digunakan adalah neo4j
2. USE di dalam kueri Cyhper 
3. garf sosial atau friendship network merepresentasikan entitas manusia (person) yang terhubung oleh relasi pertemanan ([:KNOWS])

In [7]:
import os
import sys
from argparse import ArgumentParser
from neo4j import GraphDatabase

def parse_arguments():
    parser = ArgumentParser(description="Impor dataset RetailRocket (events.csv) ke database Neo4j.")
    parser.add_argument("-u", "--user", default="neo4j", help="Username Neo4j (default: neo4j)")
    parser.add_argument("-p", "--password", default="utsgrafkoni", help="Password Neo4j (sesuaikan passwordmu)")
    parser.add_argument("-b", "--bolt", default="neo4j://127.0.0.1:7687", help="Neo4j URI / Bolt")
    parser.add_argument("-s", "--source", default="ecommerce-data", help="Direktori/folder sumber data (sesuaikan dengan nama folder hasil ekstrakmu)")
    return parser.parse_args(args=[]) # Gunakan args=[] jika dijalankan langsung di Jupyter Notebook agar tidak error argumen

args = parse_arguments()

# 1. Pemeriksaan keberadaan berkas events.csv
csv_file_path = os.path.join(args.source, "events.csv")

if not os.path.exists(csv_file_path):
    print(f"Error: File '{csv_file_path}' tidak ditemukan!")
    sys.exit(1)
else:
    print(f"File ditemukan di: {csv_file_path}")

# 2. Pembuatan koneksi ke Neo4j
try:
    driver = GraphDatabase.driver(
        args.bolt, auth=(args.user, args.password)
    )
    driver.verify_connectivity()
    print("Berhasil terhubung ke database Neo4j!")
except Exception as e:
    print(f"Gagal terhubung ke database: {e}")
    sys.exit(1)

File ditemukan di: ecommerce-data\events.csv
Berhasil terhubung ke database Neo4j!


C:\Users\Noni\AppData\Local\Temp\ipykernel_28764\2216844597.py:30: ExperimentalWarning: The configuration may change in the future.
  driver.verify_connectivity()


In [13]:
import pandas as pd
from tqdm import tqdm

# Tentukan ukuran batch
batch_size = 10000

# 1. Fungsi untuk membuat constraint
def create_constraints(tx):
    tx.run("CREATE CONSTRAINT event_id IF NOT EXISTS FOR (e:Event) REQUIRE e.timestamp IS UNIQUE")

# 2. Fungsi untuk mengeksekusi query Cypher per batch record
def import_batch(tx, records):
    query = """
    UNWIND $records AS row
    MERGE (v:Visitor {visitorid: row.visitorid})
    MERGE (i:Item {itemid: row.itemid})
    CREATE (e:Event {
        timestamp: row.timestamp,
        event: row.event,
        transactionid: row.transactionid
    })
    MERGE (v)-[:PERFORMED]->(e)
    MERGE (e)-[:ON_ITEM]->(i)
    """
    tx.run(query, records=records)

# 3. Proses Impor Utama
with driver.session() as session:
    try:
        # Menggunakan write_transaction untuk Neo4j driver versi 4.x
        session.write_transaction(create_constraints)
    except Exception as e:
        print(f"Catatan constraint: {e}")
    
    print(f"Membaca file data dari: {csv_file_path}")
    
    # Hitung total baris untuk progress bar
    total_rows = sum(1 for _ in open(csv_file_path, encoding="utf-8")) - 1
    total_chunks = (total_rows // batch_size) + 1
    
    print("Memulai proses impor data ke Neo4j...")
    for chunk in tqdm(
        pd.read_csv(csv_file_path, chunksize=batch_size), 
        total=total_chunks, 
        desc="Mengimpor Batch Data"
    ):
        # Membersihkan data null/NaN pada transactionid
        chunk["transactionid"] = (
            chunk["transactionid"].fillna("").astype(str)
        )
        
        records = chunk.to_dict(orient="records")
        # Mengubah execute_write menjadi write_transaction
        session.write_transaction(import_batch, records)

driver.close()
print("Proses impor berhasil.")

Membaca file data dari: ecommerce-data\events.csv
Memulai proses impor data ke Neo4j...


Mengimpor Batch Data:   0%|          | 0/276 [00:11<?, ?it/s]
c:\Users\Noni\Downloads\myenv\Lib\site-packages\neo4j\work\simple.py:373: SyntaxWarning: invalid escape sequence '\*'
  :param transaction_function: a function that takes a transaction as an argument and does work with the transaction. `tx_function(tx, \*args, \*\*kwargs)`
c:\Users\Noni\Downloads\myenv\Lib\site-packages\neo4j\work\simple.py:398: SyntaxWarning: invalid escape sequence '\*'
  :param transaction_function: a function that takes a transaction as an argument and does work with the transaction. `tx_function(tx, \*args, \*\*kwargs)`


ConstraintError: {code: Neo.ClientError.Schema.ConstraintValidationFailed} {message: Node(9446) already exists with label `Event` and property `timestamp` = 1433216459038}